# Deploy the graph agent with boto3

Deploys Demo 02 one resource at a time with the AWS SDK for Python: the same agent as Demo 01, running on
[Amazon Bedrock AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-how-it-works.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el) (platform version V2, code package, no container),
with Tavily behind [AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-target-integrations.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el) and the knowledge graph in Neo4j AuraDB.

```
 you ──InvokeAgentRuntime──▶ Runtime (V2, research_agent.py) ──SigV4──▶ Gateway ──API key──▶ api.tavily.com
                                  │  └─ Bedrock model                      └─ key kept in AgentCore Identity
                                  └─ Secrets Manager ──▶ Neo4j AuraDB credentials
```

| # | Resource | API | Docs |
|---|----------|-----|------|
| 2 | Secrets Manager secret (AuraDB credentials) | `create_secret` | [CreateSecret](https://docs.aws.amazon.com/secretsmanager/latest/apireference/API_CreateSecret.html) |
| 3 | AgentCore Identity API key credential provider (Tavily) | `create_api_key_credential_provider` | [Outbound auth with an API key](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-outbound-auth.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el) |
| 4 | IAM role for the Gateway | `create_role`, `put_role_policy` | [Gateway service role permissions](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-prerequisites-permissions.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el) |
| 5 | Gateway (MCP, IAM inbound) | `create_gateway` | [Create a gateway with the API](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-create-api.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el) |
| 6 | Gateway target `tavily` (OpenAPI) | `create_gateway_target` | [OpenAPI targets](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-schema-openapi.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el) |
| 7 | S3 bucket + code package | `create_bucket`, `upload_file` | [Direct code deployment](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-get-started-code-deploy-python.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el) |
| 8 | IAM execution role for the Runtime | `create_role`, `put_role_policy` | [Runtime permissions](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-permissions.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el) |
| 9 | AgentCore Runtime (V2) | `create_agent_runtime` | [Platform versions](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-how-it-works.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el) |
| 10 | Ask the agent | `invoke_agent_runtime` | [Invoke an agent runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/APIReference/API_InvokeAgentRuntime.html) |
| 11 | Cleanup, in reverse order | `delete_*` | |

**Before you run it**
- AWS credentials that can create the resources above, and a Bedrock model you can invoke (default Claude Sonnet 4.5).
- `boto3 >= 1.43.95` (the first release whose `CreateAgentRuntime` has `platformVersion`), `uv` (the build script uses it) and `zip`.
- Exported in the shell that started Jupyter: `TAVILY_API_KEY` (or `TAVILY_KEY`) and your AuraDB `NEO4J_URI`, `NEO4J_USERNAME`
  (or `NEO4J_USER`), `NEO4J_PASSWORD`, `NEO4J_DATABASE`. Secrets are read from the environment and are never printed.
- Run the cells top to bottom. If a cell fails halfway, run section 11 (it finds everything by name) and start again.
- Cost: AgentCore Runtime and Gateway are pay-per-use; Secrets Manager charges per secret per month. Run section 11 when you finish.

## 1. Configuration

Names, Region and the account (read from STS, so no account ID is written anywhere). Runtime names allow only letters,
digits and underscores; Gateway names allow hyphens. This notebook uses the prefix `tavily-graph-boto3`, so it does not
collide with the CLI notebook or the CDK stack if you try the other paths later.

`mask()` hides the account ID in everything the notebook prints. `wait_for()` and `wait_gone()` are the polling helpers:
Gateway and Runtime have no boto3 waiters, and the docs poll `get_agent_runtime` the same way
([V2 create and update take minutes](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-how-it-works.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el)).

In [ ]:
import json
import os
import subprocess
import time
import uuid
from pathlib import Path

import boto3
from botocore.exceptions import ClientError

REGION = os.environ.get("AWS_REGION", "us-east-1")
PREFIX = "tavily-graph-boto3"
MODEL_ID = os.environ.get("BEDROCK_MODEL_ID", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")

NEO4J_SECRET_NAME = f"{PREFIX}/neo4j-aura"
KEY_PROVIDER_NAME = f"{PREFIX}-tavily-key"
GATEWAY_ROLE_NAME = f"{PREFIX}-gateway-role"
GATEWAY_NAME = PREFIX
TARGET_NAME = "tavily"                      # tool names become tavily___search_web, tavily___read_web_page
RUNTIME_ROLE_NAME = f"{PREFIX}-runtime-role"
RUNTIME_NAME = "tavily_graph_boto3"
ENTRYPOINT = "research_agent.py"            # any .py file works: entryPoint is an array, main.py is only the docs' example
CODE_KEY = "research-agent/agent-code.zip"

# Secrets come from the environment and are never printed.
TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY") or os.environ.get("TAVILY_KEY")
NEO4J = {
    "uri": os.environ.get("NEO4J_URI"),
    "username": os.environ.get("NEO4J_USERNAME") or os.environ.get("NEO4J_USER") or "neo4j",
    "password": os.environ.get("NEO4J_PASSWORD"),
    "database": os.environ.get("NEO4J_DATABASE", "neo4j"),
}
missing = [name for name, value in {"TAVILY_API_KEY": TAVILY_API_KEY, "NEO4J_URI": NEO4J["uri"],
                                    "NEO4J_PASSWORD": NEO4J["password"]}.items() if not value]
assert not missing, f"export these in the shell that started Jupyter: {missing}"

account = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
BUCKET = f"{PREFIX}-code-{account}-{REGION}"


def mask(value):
    """Hide the account ID in anything we print."""
    return str(value).replace(account, "<account>")


iam = boto3.client("iam", region_name=REGION)
secrets = boto3.client("secretsmanager", region_name=REGION)
s3 = boto3.client("s3", region_name=REGION)
control = boto3.client("bedrock-agentcore-control", region_name=REGION)    # create and manage resources
runtime_api = boto3.client("bedrock-agentcore", region_name=REGION)        # invoke the agent


def wait_for(describe, what, ready="READY", timeout=1500, interval=10):
    """Poll describe() until its status is `ready`. Stop on a status that ends in FAILED or UNSUCCESSFUL."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        response = describe()
        status = response["status"]
        print(f"  {what}: {status}")
        if status == ready:
            return response
        if status.endswith(("FAILED", "UNSUCCESSFUL")):
            raise RuntimeError(f"{what} ended in {status}: {response.get('failureReason') or response.get('statusReasons')}")
        time.sleep(interval)
    raise TimeoutError(f"{what} was not {ready} after {timeout} s")


def wait_gone(describe, what, timeout=900, interval=10):
    """Poll describe() until it raises ResourceNotFoundException (the resource is deleted)."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            print(f"  {what}: {describe()['status']}")
        except ClientError as error:
            if error.response["Error"]["Code"] == "ResourceNotFoundException":
                print(f"  {what}: deleted")
                return
            raise
        time.sleep(interval)
    raise TimeoutError(f"{what} still exists after {timeout} s")


def trust_policy(source_arn_pattern):
    """Trust policy from the AgentCore docs: only AgentCore, only from this account and these resources."""
    return json.dumps({
        "Version": "2012-10-17",
        "Statement": [{
            "Sid": "AssumeRolePolicy",
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
            "Condition": {"StringEquals": {"aws:SourceAccount": account},
                          "ArnLike": {"aws:SourceArn": source_arn_pattern}},
        }],
    })


print("Region:", REGION, "| account:", mask(account), "| code bucket:", mask(BUCKET))

## 2. Secrets Manager secret for the AuraDB credentials

**What:** a secret with `{"uri", "username", "password", "database"}`. **Why:** the agent reads it at startup (`NEO4J_SECRET_ID`),
so the password is never in the code package, the runtime environment variables or a template. The runtime role (section 8)
gets `secretsmanager:GetSecretValue` on this one secret.
Docs: [CreateSecret](https://docs.aws.amazon.com/secretsmanager/latest/apireference/API_CreateSecret.html).

In [ ]:
neo4j_secret = secrets.create_secret(
    Name=NEO4J_SECRET_NAME,
    Description="Neo4j AuraDB connection for the graph agent demo",
    SecretString=json.dumps(NEO4J),
)
neo4j_secret_arn = neo4j_secret["ARN"]
print("Secret created:", mask(neo4j_secret_arn))

## 3. AgentCore Identity API key credential provider for Tavily

**What:** a credential provider in AgentCore Identity that stores the Tavily API key. **Why:** the Gateway target (section 6)
references the provider by ARN, and the Gateway adds the key to each request to `api.tavily.com`. The agent and its
container never see the Tavily key. AgentCore keeps the key in a secret it manages, and returns that secret's ARN, which
the Gateway role needs in section 4.
Docs: [Set up outbound authorization with an API key](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-outbound-auth.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el).

In [ ]:
provider = control.create_api_key_credential_provider(name=KEY_PROVIDER_NAME, apiKey=TAVILY_API_KEY)
provider_arn = provider["credentialProviderArn"]
provider_secret_arn = provider["apiKeySecretArn"]["secretArn"]
print("Credential provider created:", mask(provider_arn))

## 4. IAM role for the Gateway

**What:** the service role the Gateway assumes. The trust policy lets only `bedrock-agentcore.amazonaws.com` assume it, and only
for gateways in this account whose ID starts with the gateway name (a gateway ID is `<name>-<random suffix>`, so the exact ARN does
not exist yet). **Why:** to add the API key to the outbound call the Gateway must read it. The docs list three permissions for
API key outbound auth: `GetWorkloadAccessToken` (on the workload identity of this gateway), `GetResourceApiKey` (on the credential
provider) and `secretsmanager:GetSecretValue` (on the secret AgentCore created for the key).
Docs: [Gateway service role](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-prerequisites-permissions.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el), [API key outbound auth](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-outbound-auth.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el).

IAM is eventually consistent, so the cell waits a few seconds before the next section uses the role
([IAM troubleshooting](https://docs.aws.amazon.com/IAM/latest/UserGuide/troubleshoot_general.html#troubleshoot_general_eventual-consistency)).

In [ ]:
gateway_role = iam.create_role(
    RoleName=GATEWAY_ROLE_NAME,
    Description="AgentCore Gateway service role for the Tavily target (graph agent demo)",
    AssumeRolePolicyDocument=trust_policy(f"arn:aws:bedrock-agentcore:{REGION}:{account}:gateway/{GATEWAY_NAME}-*"),
)
gateway_role_arn = gateway_role["Role"]["Arn"]

iam.put_role_policy(
    RoleName=GATEWAY_ROLE_NAME,
    PolicyName="tavily-api-key-outbound-auth",
    PolicyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [
            {"Sid": "GetWorkloadAccessToken", "Effect": "Allow",
             "Action": "bedrock-agentcore:GetWorkloadAccessToken",
             "Resource": [f"arn:aws:bedrock-agentcore:{REGION}:{account}:workload-identity-directory/default",
                          f"arn:aws:bedrock-agentcore:{REGION}:{account}:workload-identity-directory/default/workload-identity/{GATEWAY_NAME}-*"]},
            {"Sid": "GetResourceApiKey", "Effect": "Allow",
             "Action": "bedrock-agentcore:GetResourceApiKey",
             "Resource": [f"arn:aws:bedrock-agentcore:{REGION}:{account}:token-vault/default",
                          provider_arn]},
            {"Sid": "GetSecretValue", "Effect": "Allow",
             "Action": "secretsmanager:GetSecretValue",
             "Resource": provider_secret_arn},
        ],
    }),
)
print("Gateway role created:", mask(gateway_role_arn))
time.sleep(10)

## 5. Gateway

**What:** an MCP gateway with `AWS_IAM` inbound authorization. **Why:** the agent calls it with its own IAM credentials (SigV4), so
there is no OAuth server to run; callers need `bedrock-agentcore:InvokeGateway` on this gateway (granted in section 8).
`create_gateway` returns while the gateway is `CREATING`, so the cell polls `get_gateway` until `READY`.
Docs: [Create a gateway with the API](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-create-api.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el), [inbound authorization](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-inbound-auth.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el).

In [ ]:
gateway = control.create_gateway(
    name=GATEWAY_NAME,
    description="Tavily web search and extract as MCP tools",
    roleArn=gateway_role_arn,
    protocolType="MCP",
    authorizerType="AWS_IAM",
)
gateway_id = gateway["gatewayId"]
gateway = wait_for(lambda: control.get_gateway(gatewayIdentifier=gateway_id), "gateway")
gateway_arn, gateway_url = gateway["gatewayArn"], gateway["gatewayUrl"]
print("Gateway ready:", mask(gateway_url))

## 6. Gateway target `tavily`

**What:** an OpenAPI target built from `schemas/tavily-openapi.json`, with the API key from section 3 sent as the header
`Authorization: Bearer <key>`. **Why:** AgentCore documents a *Tavily Search* integration template (server `https://api.tavily.com`,
API key auth, `/search` and `/extract`), but only in the console
([integration templates](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-target-integrations.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el)). The API has no template field, so this target describes the same
server as an OpenAPI schema. We write the operations ourselves (`search_web`, `read_web_page`): the model reads each
`operationId` and `description` as the tool's name and description on every call, so they say when to use the tool and
when not, and match the local notebook's `@tool` functions. The schema follows the [OpenAPI target rules](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-schema-openapi.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el):
`operationId` on every operation, JSON only, no `oneOf`/`anyOf`/`allOf`, no security schemes (auth lives in the credential configuration).
The Gateway names each tool `<target>___<operationId>` ([tool naming](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-tool-naming.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el)), which is what the agent's
tool filter `^tavily___` expects.

In [ ]:
openapi_schema = Path("schemas/tavily-openapi.json").read_text()

target = control.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name=TARGET_NAME,
    description="Web tools: search_web (/search) and read_web_page (/extract)",
    targetConfiguration={"mcp": {"openApiSchema": {"inlinePayload": openapi_schema}}},
    credentialProviderConfigurations=[{
        "credentialProviderType": "API_KEY",
        "credentialProvider": {"apiKeyCredentialProvider": {
            "providerArn": provider_arn,
            "credentialLocation": "HEADER",
            "credentialParameterName": "Authorization",
            "credentialPrefix": "Bearer ",
        }},
    }],
)
target_id = target["targetId"]
wait_for(lambda: control.get_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id), "target")

## 7. Code package and S3 bucket

**What:** `build_package.sh` installs the agent's dependencies as **linux/arm64** wheels
(`uv pip install --python-platform aarch64-manylinux2014 --python-version 3.12 --only-binary=:all:`), copies `agent/*.py` next to them
and zips the result (limits: 250 MB zipped, 750 MB unzipped). No Docker. The zip goes to a private S3 bucket (all public access blocked),
because `CreateAgentRuntime` reads the code from S3. **Why a bucket per account and Region:** the name follows the docs' pattern,
and `ExpectedBucketOwner` makes the upload fail if the bucket belongs to someone else.
Docs: [Direct code deployment for Python](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-get-started-code-deploy-python.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el).

In [ ]:
subprocess.run(["bash", "build_package.sh"], check=True)
CODE_ZIP = Path("build/agent-code.zip")

# us-east-1 takes no LocationConstraint; every other Region requires one.
s3.create_bucket(Bucket=BUCKET, **({} if REGION == "us-east-1" else {"CreateBucketConfiguration": {"LocationConstraint": REGION}}))
s3.put_public_access_block(
    Bucket=BUCKET,
    PublicAccessBlockConfiguration={"BlockPublicAcls": True, "IgnorePublicAcls": True,
                                    "BlockPublicPolicy": True, "RestrictPublicBuckets": True},
)
s3.upload_file(str(CODE_ZIP), BUCKET, CODE_KEY, ExtraArgs={"ExpectedBucketOwner": account})
print(f"Uploaded {CODE_ZIP.stat().st_size / 1e6:.0f} MB to s3://<bucket>/{CODE_KEY}")

## 8. IAM execution role for the Runtime

**What:** the role the agent runs as. It starts from the docs' *direct deploy execution role* (CloudWatch Logs, X-Ray, CloudWatch metrics, Bedrock model
invocation) and adds only what this agent calls: `s3:GetObject` on the one code object, `bedrock-agentcore:InvokeGateway` on the one gateway,
and `secretsmanager:GetSecretValue` on the one AuraDB secret. The model statement allows foundation models and this account's inference profiles,
because Claude Sonnet 4.5 is invoked through a `us.` inference profile. The agent does not use AgentCore Identity tokens itself (the Gateway
does), so the workload-identity statement from the full role is left out.
Docs: [Runtime permissions](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-permissions.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el).

In [ ]:
runtime_role = iam.create_role(
    RoleName=RUNTIME_ROLE_NAME,
    Description="AgentCore Runtime execution role for the graph agent demo",
    AssumeRolePolicyDocument=trust_policy(f"arn:aws:bedrock-agentcore:{REGION}:{account}:*"),
)
runtime_role_arn = runtime_role["Role"]["Arn"]
log_groups = f"arn:aws:logs:{REGION}:{account}:log-group:/aws/bedrock-agentcore/runtimes"

iam.put_role_policy(
    RoleName=RUNTIME_ROLE_NAME,
    PolicyName="graph-agent-runtime",
    PolicyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [
            {"Effect": "Allow", "Action": ["logs:DescribeLogStreams", "logs:CreateLogGroup"], "Resource": f"{log_groups}/*"},
            {"Effect": "Allow", "Action": "logs:PutResourcePolicy", "Resource": f"{log_groups}/{RUNTIME_NAME}-*"},
            {"Effect": "Allow", "Action": "logs:DescribeLogGroups", "Resource": f"arn:aws:logs:{REGION}:{account}:log-group:*"},
            {"Effect": "Allow", "Action": ["logs:CreateLogStream", "logs:PutLogEvents"], "Resource": f"{log_groups}/*:log-stream:*"},
            {"Effect": "Allow", "Action": ["xray:PutTraceSegments", "xray:PutTelemetryRecords",
                                           "xray:GetSamplingRules", "xray:GetSamplingTargets"], "Resource": "*"},
            {"Effect": "Allow", "Action": "cloudwatch:PutMetricData", "Resource": "*",
             "Condition": {"StringEquals": {"cloudwatch:namespace": "bedrock-agentcore"}}},
            {"Sid": "BedrockModelInvocation", "Effect": "Allow",
             "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"],
             "Resource": ["arn:aws:bedrock:*::foundation-model/*", f"arn:aws:bedrock:{REGION}:{account}:inference-profile/*"]},
            # What this agent adds to the docs' role:
            {"Sid": "ReadCodePackage", "Effect": "Allow", "Action": "s3:GetObject", "Resource": f"arn:aws:s3:::{BUCKET}/{CODE_KEY}"},
            {"Sid": "InvokeTavilyGateway", "Effect": "Allow", "Action": "bedrock-agentcore:InvokeGateway", "Resource": gateway_arn},
            {"Sid": "ReadNeo4jSecret", "Effect": "Allow", "Action": "secretsmanager:GetSecretValue", "Resource": neo4j_secret_arn},
        ],
    }),
)
print("Runtime role created:", mask(runtime_role_arn))
time.sleep(10)

## 9. AgentCore Runtime (platform version V2)

**What:** the runtime, from the code package. `entryPoint` is the command to run: `opentelemetry-instrument` (ADOT, for AgentCore Observability) and then the file, so the file can be named `research_agent.py`;
`main.py` is only the example name in the docs. `platformVersion="V2"` starts the agent from a snapshot taken after startup (V1 is the default if
you omit it). **Why V2 changes the code:** work done at startup is frozen in the snapshot, so `research_agent.py` builds the model, the Neo4j
driver and the graph store at module level, and builds the Gateway client and the `Agent` inside the request handler
([Optimize your agent for V2](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-v2-optimize.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el)). The agent must report healthy within 120 s of starting.

`create_agent_runtime` returns while the runtime is `CREATING`. A V2 create **takes several minutes** (the docs say to poll `get_agent_runtime`;
calling update or delete early returns `ConflictException`). Network mode `PUBLIC` lets the agent reach the Gateway, Secrets Manager and AuraDB.
The lifecycle settings keep their defaults (15 minutes idle, 8 hours maximum session).
Docs: [Platform versions](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-how-it-works.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el), [Direct code deployment](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-get-started-code-deploy-python.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el).

In [ ]:
runtime = control.create_agent_runtime(
    agentRuntimeName=RUNTIME_NAME,
    description="Research agent with a Neo4j knowledge graph; Tavily through AgentCore Gateway",
    agentRuntimeArtifact={"codeConfiguration": {
        "code": {"s3": {"bucket": BUCKET, "prefix": CODE_KEY}},
        "runtime": "PYTHON_3_12",
        "entryPoint": ["opentelemetry-instrument", ENTRYPOINT],   # ADOT for AgentCore Observability
    }},
    roleArn=runtime_role_arn,
    networkConfiguration={"networkMode": "PUBLIC"},
    platformVersion="V2",
    environmentVariables={
        "TAVILY_GATEWAY_URL": gateway_url,
        "NEO4J_SECRET_ID": NEO4J_SECRET_NAME,
        "BEDROCK_MODEL_ID": MODEL_ID,
    },
)
runtime_id = runtime["agentRuntimeId"]
runtime = wait_for(lambda: control.get_agent_runtime(agentRuntimeId=runtime_id), "runtime", timeout=1800, interval=15)
runtime_arn = runtime["agentRuntimeArn"]
print("Platform version:", runtime["platformVersion"], "| runtime:", mask(runtime_arn))

## 10. Ask the agent

**What:** `invoke_agent_runtime` with `{"prompt": ...}`. **Why a new `runtimeSessionId` per question:** each session is its own microVM, so nothing the
agent "remembers" survives between questions except the Neo4j graph. That is the point of the demo: the first questions fill the graph, the
later ones are answered from it. The session ID must be at least 33 characters, so a UUID string works. The first call after creation can
take longer (it restores the snapshot and opens the Gateway connection).
The five questions come from `invoke_agent.py`. Docs: [InvokeAgentRuntime](https://docs.aws.amazon.com/bedrock-agentcore/latest/APIReference/API_InvokeAgentRuntime.html).

Expect `tools_used` to contain `tavily___search_web` for the first questions and fewer or no searches for the last ones. Search counts are
model behavior and change from run to run.

In [ ]:
from invoke_agent import SERIES   # the five demo questions


def ask(prompt):
    response = runtime_api.invoke_agent_runtime(
        agentRuntimeArn=runtime_arn,
        runtimeSessionId=str(uuid.uuid4()),          # new session per question: the graph is the only memory
        contentType="application/json",
        accept="application/json",
        payload=json.dumps({"prompt": prompt}).encode(),
    )
    return json.loads(response["response"].read().decode())


for number, question in enumerate(SERIES, 1):
    result = ask(question)
    print(f"\nQ{number}: {question}\n  tools used: {result['tools_used']}\n  {result['answer']}")

## 11. Cleanup

Deletes everything this notebook created, **in reverse order**: runtime, gateway target, gateway, credential provider, runtime role,
gateway role, code object and bucket, secret. It finds each resource by name, so it works after a restart or a half-finished run.
Your AuraDB instance and the graph in it are not touched (`WebFactsGraph.clear()` removes the demo's nodes if you want them gone).

Nothing is deleted until you set `CONFIRM_DELETE = True` and run the cells below. A runtime deletes asynchronously, and V2 can keep its
snapshot for up to 8 hours after deletion ([snapshot lifecycle](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-how-it-works.html?trk=87c4c426-cddf-4799-a299-273337552ad8&sc_channel=el)); that is managed by AgentCore and not billed as a resource you own.
Run section 1 first if the kernel was restarted.

In [ ]:
CONFIRM_DELETE = False      # set to True to delete the resources of this notebook


def find_runtime_id():
    for page in control.get_paginator("list_agent_runtimes").paginate():
        for item in page["agentRuntimes"]:
            if item["agentRuntimeName"] == RUNTIME_NAME:
                return item["agentRuntimeId"]


def find_gateway_id():
    for page in control.get_paginator("list_gateways").paginate():
        for item in page["items"]:
            if item["name"] == GATEWAY_NAME:
                return item["gatewayId"]


def delete_role(role_name):
    try:
        for policy_name in iam.list_role_policies(RoleName=role_name)["PolicyNames"]:
            iam.delete_role_policy(RoleName=role_name, PolicyName=policy_name)
        iam.delete_role(RoleName=role_name)
        print("Deleted role", role_name)
    except iam.exceptions.NoSuchEntityException:
        print("Role already gone:", role_name)


print("Deleting resources" if CONFIRM_DELETE else "CONFIRM_DELETE is False: the cells below only report what they would delete.")

In [ ]:
# 1. Runtime
runtime_id = find_runtime_id()
if CONFIRM_DELETE and runtime_id:
    control.delete_agent_runtime(agentRuntimeId=runtime_id)
    wait_gone(lambda: control.get_agent_runtime(agentRuntimeId=runtime_id), "runtime")
else:
    print("runtime:", "would delete" if runtime_id else "not found")

In [ ]:
# 2. Gateway targets, then the gateway
gateway_id = find_gateway_id()
if CONFIRM_DELETE and gateway_id:
    for target in control.list_gateway_targets(gatewayIdentifier=gateway_id)["items"]:
        control.delete_gateway_target(gatewayIdentifier=gateway_id, targetId=target["targetId"])
        wait_gone(lambda: control.get_gateway_target(gatewayIdentifier=gateway_id, targetId=target["targetId"]), "target")
    control.delete_gateway(gatewayIdentifier=gateway_id)
    wait_gone(lambda: control.get_gateway(gatewayIdentifier=gateway_id), "gateway")
else:
    print("gateway and targets:", "would delete" if gateway_id else "not found")

In [ ]:
# 3. Credential provider (AgentCore Identity)
if CONFIRM_DELETE:
    try:
        control.delete_api_key_credential_provider(name=KEY_PROVIDER_NAME)
        print("Deleted credential provider", KEY_PROVIDER_NAME)
    except ClientError as error:
        print("Credential provider:", error.response["Error"]["Code"])
else:
    print("credential provider: would delete", KEY_PROVIDER_NAME)

In [ ]:
# 4. IAM roles (inline policies first)
if CONFIRM_DELETE:
    delete_role(RUNTIME_ROLE_NAME)
    delete_role(GATEWAY_ROLE_NAME)
else:
    print("roles: would delete", RUNTIME_ROLE_NAME, "and", GATEWAY_ROLE_NAME)

In [ ]:
# 5. Code package and bucket
if CONFIRM_DELETE:
    try:
        s3.delete_object(Bucket=BUCKET, Key=CODE_KEY, ExpectedBucketOwner=account)
        s3.delete_bucket(Bucket=BUCKET, ExpectedBucketOwner=account)
        print("Deleted bucket and code package")
    except ClientError as error:
        print("Bucket:", error.response["Error"]["Code"])
else:
    print("bucket: would delete", mask(BUCKET))

In [ ]:
# 6. AuraDB secret (no recovery window, so the name can be reused at once)
if CONFIRM_DELETE:
    try:
        secrets.delete_secret(SecretId=NEO4J_SECRET_NAME, ForceDeleteWithoutRecovery=True)
        print("Deleted secret", NEO4J_SECRET_NAME)
    except secrets.exceptions.ResourceNotFoundException:
        print("Secret already gone")
else:
    print("secret: would delete", NEO4J_SECRET_NAME)